In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, GlobalAveragePooling2D, GlobalMaxPooling2D, Dense, Add, Multiply, Activation, Reshape, Flatten
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

In [3]:
import albumentations as A

# Data processing
img_data_list = []
labels = []

# Define augmentation techniques
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(p=0.2)
])

# Function to process and augment images
def process_images(data_path, label, save_path, num_aug=3):
    os.makedirs(save_path, exist_ok=True)  # Create folder for augmented images

    for dataset in os.listdir(data_path):
        img_path = os.path.join(data_path, dataset)
        input_img = cv2.imread(img_path)

        if input_img is None:
            print(f"Warning: Image {dataset} could not be loaded.")
            continue

        input_img = cv2.cvtColor(input_img, cv2.COLOR_BGR2RGB)
        input_img_resize = cv2.resize(input_img, (224, 224))
        input_img_resize = cv2.normalize(input_img_resize, None, 0, 255, cv2.NORM_MINMAX)

        # Add original image to dataset
        img_data_list.append(input_img_resize)
        labels.append(label)

        # Augment and add multiple versions
        for i in range(num_aug):
            augmented = transform(image=input_img_resize)['image']
            img_data_list.append(augmented)  # Add augmented image
            labels.append(label)  # Keep same label

            # Save augmented image
            aug_filename = f"aug_{i}_{dataset}"
            cv2.imwrite(os.path.join(save_path, aug_filename), cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))

# Load and augment data for each class
process_images('/content/drive/MyDrive/major-project/dataset/0-normal/0', 0,
               '/content/drive/MyDrive/major-project/dataset/0-normal/0_Augmented')

process_images('/content/drive/MyDrive/major-project/dataset/1-oesteopenia/1', 1,
               '/content/drive/MyDrive/major-project/dataset/1-oesteopenia/1_Augmented')

process_images('/content/drive/MyDrive/major-project/dataset/2-oestopororsis/2', 2,
               '/content/drive/MyDrive/major-project/dataset/2-oestopororsis/2_Augmented')

In [4]:
print(f"Total images in dataset: {len(img_data_list)}")
print(f"Total labels in dataset: {len(labels)}")

# Optional: Count images per class
unique_labels, counts = np.unique(labels, return_counts=True)
for label, count in zip(unique_labels, counts):
    print(f"Class {label}: {count} images")


Total images in dataset: 7788
Total labels in dataset: 7788
Class 0: 3120 images
Class 1: 1496 images
Class 2: 3172 images


In [5]:
# from keras.utils import np_utils
from keras.utils import to_categorical
num_classes = 3
label = to_categorical(labels, num_classes)

In [6]:
# Convert lists to arrays
img_data = np.array(img_data_list)
labels = np.array(labels)

print(f'Image data shape: {img_data.shape}')
print(f'Labels shape: {labels.shape}')


Image data shape: (7788, 224, 224, 3)
Labels shape: (7788,)


In [7]:
print(len(img_data))  # Should be > 0
print(len(label))  # Should be > 0


7788
7788


In [8]:
# One-hot encode the labels
labels = to_categorical(labels, num_classes=3)

In [9]:
print(f'Labels shape after one-hot encoding: {labels.shape}')

Labels shape after one-hot encoding: (7788, 3)


In [10]:
# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(img_data, labels, test_size=0.2, random_state=42)

In [11]:
print(f'X_train shape after split: {X_train.shape}')
print(f'y_train shape after split: {y_train.shape}')
print(f'X_val shape after split: {X_val.shape}')
print(f'y_val shape after split: {y_val.shape}')

X_train shape after split: (6230, 224, 224, 3)
y_train shape after split: (6230, 3)
X_val shape after split: (1558, 224, 224, 3)
y_val shape after split: (1558, 3)


In [12]:
assert X_train.shape[0] == y_train.shape[0], "Mismatch in number of samples between X_train and y_train"
assert X_val.shape[0] == y_val.shape[0], "Mismatch in number of samples between X_val and y_val"

In [13]:
def cam_block(x):
    print("Input shape:", x.shape)

    # Apply convolution with dilation rates
    dilated_1 = Conv2D(x.shape[-1], (3, 3), activation='relu', dilation_rate=1, padding='same')(x)
    print("dilated_1 shape:", dilated_1.shape)

    dilated_2 = Conv2D(x.shape[-1], (3, 3), activation='relu', dilation_rate=2, padding='same')(x)
    print("dilated_2 shape:", dilated_2.shape)

    dilated_3 = Conv2D(x.shape[-1], (3, 3), activation='relu', dilation_rate=3, padding='same')(x)
    print("dilated_3 shape:", dilated_3.shape)

    # Element-wise add conv_1 and conv_2
    add_result = Add()([dilated_1, dilated_2])
    print("add_result shape:", add_result.shape)

    # Element-wise multiply conv_2 and conv_3
    mul_result = Multiply()([dilated_2, dilated_3])
    print("mul_result shape:", mul_result.shape)

    # 1x1 convolutions on the results
    conv_add = Conv2D(x.shape[-1], (1, 1), activation='relu', padding='same')(add_result)
    print("conv_add shape:", conv_add.shape)
    conv_mul = Conv2D(x.shape[-1], (1, 1), activation='relu', padding='same')(mul_result)
    print("conv_mul shape:", conv_mul.shape)

    # Apply Global Average Pooling and Global Max Pooling
    gap = GlobalAveragePooling2D()(conv_add)
    print("gap shape:", gap.shape)
    gmp = GlobalMaxPooling2D()(conv_mul)
    print("gmp shape:", gmp.shape)

    # Multiply the pooled outputs
    combined = Multiply()([gap, gmp])
    print("combined shape:", combined.shape)

    # Pass through sigmoid activation function
    sigmoid_output = Activation('sigmoid')(combined)
    print("sigmoid_output shape:", sigmoid_output.shape)

    # Multiply with the initial input
    multiplied_output = Multiply()([x, sigmoid_output])
    print("multiplied_output shape:", multiplied_output.shape)

    # Element-wise add the multiplied output to the initial input
    final_output = Add()([x, multiplied_output])
    print("final_output shape:", final_output.shape)

    return final_output

In [14]:
from tensorflow.keras.layers import Concatenate

# Define the main model with CAM blocks
input_shape = (224, 224, 3)
num_classes = 3

inputs = Input(shape=input_shape)

# First convolutional layer with dilation rate 1
x1 = Conv2D(32, (3, 3), activation='relu', dilation_rate=1, padding='same')(inputs)

# Second convolutional layer with dilation rate 2
x2 = Conv2D(32, (3, 3), activation='relu', dilation_rate=2, padding='same')(x1)

# Third convolutional layer with dilation rate 3
x3 = Conv2D(64, (3, 3), activation='relu', dilation_rate=3, padding='same')(x2)

# Fourth convolutional layer with dilation rate 4
x4 = Conv2D(64, (3, 3), activation='relu', dilation_rate=4, padding='same')(x3)

# Fifth convolutional layer with dilation rate 5
x5 = Conv2D(64, (3, 3), activation='relu', dilation_rate=5, padding='same')(x4)

# Apply the CAM block to each convolutional output
cam_output1 = cam_block(x1)
cam_output2 = cam_block(x2)
cam_output3 = cam_block(x3)
cam_output4 = cam_block(x4)
cam_output5 = cam_block(x5)

# Apply Global Max Pooling to each CAM output
gmp1 = GlobalMaxPooling2D()(cam_output1)
gmp2 = GlobalMaxPooling2D()(cam_output2)
gmp3 = GlobalMaxPooling2D()(cam_output3)
gmp4 = GlobalMaxPooling2D()(cam_output4)
gmp5 = GlobalMaxPooling2D()(cam_output5)

# Feature aggregation layer
aggregated_features = Concatenate()([gmp1, gmp2, gmp3, gmp4, gmp5])

# Add a Dense layer with 32 units
dense_features = Dense(32, activation='relu')(aggregated_features)

# Add another Dense layer with 2 units for classification
dense_output = Dense(num_classes, activation='softmax')(dense_features)

# Create the model
model = Model(inputs, dense_output)


Input shape: (None, 224, 224, 32)
dilated_1 shape: (None, 224, 224, 32)
dilated_2 shape: (None, 224, 224, 32)
dilated_3 shape: (None, 224, 224, 32)
add_result shape: (None, 224, 224, 32)
mul_result shape: (None, 224, 224, 32)
conv_add shape: (None, 224, 224, 32)
conv_mul shape: (None, 224, 224, 32)
gap shape: (None, 32)
gmp shape: (None, 32)
combined shape: (None, 32)
sigmoid_output shape: (None, 32)
multiplied_output shape: (None, 224, 224, 32)
final_output shape: (None, 224, 224, 32)
Input shape: (None, 224, 224, 32)
dilated_1 shape: (None, 224, 224, 32)
dilated_2 shape: (None, 224, 224, 32)
dilated_3 shape: (None, 224, 224, 32)
add_result shape: (None, 224, 224, 32)
mul_result shape: (None, 224, 224, 32)
conv_add shape: (None, 224, 224, 32)
conv_mul shape: (None, 224, 224, 32)
gap shape: (None, 32)
gmp shape: (None, 32)
combined shape: (None, 32)
sigmoid_output shape: (None, 32)
multiplied_output shape: (None, 224, 224, 32)
final_output shape: (None, 224, 224, 32)
Input shape: (None

In [15]:
from tensorflow.keras import backend as K

# Clear the session to free up memory
K.clear_session()


In [ ]:
# Check the shapes of the training and validation data
print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'X_val shape: {X_val.shape}')
print(f'y_val shape: {y_val.shape}')

# The shapes should match in the first dimension (number of samples)
# Ensure all arrays contain the same number of samples
assert X_train.shape[0] == y_train.shape[0], "Mismatch in number of samples between X_train and y_train"
assert X_val.shape[0] == y_val.shape[0], "Mismatch in number of samples between X_val and y_val"

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train,
                    epochs=50,
                    batch_size=16,
                    validation_data=(X_val, y_val))


X_train shape: (6230, 224, 224, 3)
y_train shape: (6230, 3)
X_val shape: (1558, 224, 224, 3)
y_val shape: (1558, 3)
Epoch 1/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 480s 1s/step - accuracy: 0.5257 - loss: 2.5961 - val_accuracy: 0.5976 - val_loss: 1.0551
Epoch 2/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 411s 988ms/step - accuracy: 0.6169 - loss: 1.0936 - val_accuracy: 0.6451 - val_loss: 0.9823
Epoch 3/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 442s 987ms/step - accuracy: 0.6562 - loss: 0.8624 - val_accuracy: 0.7073 - val_loss: 0.6924
Epoch 4/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 426s 947ms/step - accuracy: 0.6934 - loss: 0.7466 - val_accuracy: 0.7118 - val_loss: 0.6961
Epoch 5/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 384s 984ms/step - accuracy: 0.7073 - loss: 0.7182 - val_accuracy: 0.7028 - val_loss: 0.7668
Epoch 6/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 443s 986ms/step - accuracy: 0.7236 - loss: 0.6511 - val_accuracy: 0.7522 - val_loss: 0.5692
Epoch 7/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 442s 986ms/step - accuracy: 0.7557 - loss: 0.5839 - val